# Model Agreement & Uncertainty Analysis: Per-Patient Disagreement Flagging

This notebook implements per-patient model agreement analysis for heart disease prediction:
1. **Per-Patient Model Agreement Scoring**: Measures consensus/disagreement among the base classifiers (Random Forest, XGBoost, AdaBoost) for each individual patient using probability standard deviation.
2. **Plain-Language Disagreement Flags**: Generates actionable, patient-specific risk flags (e.g., *"Model agreement: low — RF: 31%, XGBoost: 75%, AdaBoost: 58%. Interpret this prediction with extra caution."*).
3. **Accuracy Comparison by Agreement Level**: Evaluates whether model disagreement correlates with lower prediction accuracy across out-of-fold predictions.


In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Ensure src module is in python path
sys.path.append('..')

from src.preprocessing import load_data
from src.evaluation import evaluate_nested_cv

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 1. Execute Out-of-Fold Nested Cross-Validation Pipeline
Run 5-fold outer / 3-fold inner nested CV. Base model probabilities (RF, XGBoost, AdaBoost) are collected per patient across out-of-fold test sets without data leakage.


In [2]:
raw_data_path = '../' + config['dataset']['raw_path']
df = load_data(raw_data_path)

print("Running nested CV pipeline with model agreement tracking...")
results = evaluate_nested_cv(
    df,
    target_col='target',
    outer_splits=config['cv']['outer_folds'],
    inner_splits=config['cv']['inner_folds'],
    n_trials=config['cv']['optuna_n_trials'],
    random_state=config['random_state']
)

agreement_data = results['agreement_analysis']
df_patients = agreement_data['patient_df']
summary_table = agreement_data['summary_table']

print("Nested CV & agreement analysis completed.")


Running nested CV pipeline with model agreement tracking...


Nested CV & agreement analysis completed.


## 2. Distribution of Model Agreement Scores
Plot the distribution of per-patient probability standard deviations across all Cleveland dataset instances.


In [3]:
plt.figure(figsize=(9, 5))
sns.histplot(df_patients['std_dev'], bins=20, kde=True, color='purple', edgecolor='black')
plt.axvline(x=0.05, color='green', linestyle='--', label='High Agreement Boundary (std < 0.05)')
plt.axvline(x=0.15, color='red', linestyle='--', label='Low Agreement Boundary (std >= 0.15)')

plt.xlabel('Base Model Probability Std Dev', fontsize=12)
plt.ylabel('Patient Count', fontsize=12)
plt.title('Distribution of Per-Patient Model Agreement Scores (Std Dev across RF, XGB, Ada)', fontsize=14)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()

results_dir = '../' + config['results_dir']
plt.savefig(f"{results_dir}/model_agreement_distribution.png", dpi=300)
plt.show()


## 3. Agreement Level Summary & Accuracy Breakdown
Compare prediction accuracy across High, Moderate, and Low agreement groups to verify if disagreement corresponds to reduced model reliability.


In [4]:
print("=== Model Agreement Summary Breakdown ===")
display(summary_table)


=== Model Agreement Summary Breakdown ===


,Agreement Level,Sample Size (N),% of Total Patients,Accuracy
0,high agreement,46,15.2%,0.782609
1,moderate agreement,255,84.2%,0.862745
2,low agreement,2,0.7%,0.500000


## 4. Patient Examples Across Agreement Levels
Inspect specific patient examples with their individual base model probabilities and generated plain-language flag strings.


In [5]:
print("=== Patient Examples by Agreement Level ===")

for level in ['high agreement', 'moderate agreement', 'low agreement']:
    sample = df_patients[df_patients['agreement_level'] == level].head(2)
    count = len(df_patients[df_patients['agreement_level'] == level])
    print("\n--- Category: " + level.upper() + " (N = " + str(count) + ") ---")
    for idx, row in sample.iterrows():
        print("Patient ID: " + str(row['patient_index']) + " | Actual Target: " + str(row['y_true']) + " | Ensemble Pred: " + str(row['y_pred']))
        print("  Flag String: " + str(row['flag_string']) + "\n")


=== Patient Examples by Agreement Level ===

--- Category: HIGH AGREEMENT (N = 46) ---
Patient ID: 3 | Actual Target: 1 | Ensemble Pred: 1
  Flag String: Model agreement: high agreement — RF: 66%, XGBoost: 58%, AdaBoost: 61%. High consensus among base classifiers.

Patient ID: 13 | Actual Target: 0 | Ensemble Pred: 0
  Flag String: Model agreement: high agreement — RF: 51%, XGBoost: 46%, AdaBoost: 44%. High consensus among base classifiers.


--- Category: MODERATE AGREEMENT (N = 255) ---
Patient ID: 0 | Actual Target: 1 | Ensemble Pred: 1
  Flag String: Model agreement: moderate agreement — RF: 94%, XGBoost: 96%, AdaBoost: 83%. Moderate variance among base classifiers.

Patient ID: 1 | Actual Target: 1 | Ensemble Pred: 1
  Flag String: Model agreement: moderate agreement — RF: 57%, XGBoost: 77%, AdaBoost: 68%. Moderate variance among base classifiers.


--- Category: LOW AGREEMENT (N = 2) ---
Patient ID: 101 | Actual Target: 0 | Ensemble Pred: 1
  Flag String: Model agreement: low agr